# Graphs, Linear-Time Sorting, and the Toolchain

A **graph** is the most general data structure in this book: vertices, and edges between them. Roads, prerequisites, friendships, package dependencies, web links, and the layers of a neural network are all graphs, and the same half-dozen algorithms answer questions about all of them. This lab builds those algorithms and — as importantly — builds the cases where they *fail*, because "Dijkstra is wrong here" is the thing you need to recognise in production.

Then two things that are not graphs but belong in the same toolbox: sorting in **linear** time by refusing to compare, and the working tools you reach for daily — regular expressions and streaming pipelines. Regex is fully runnable here (`re` is standard library), so every pattern below is executed against real strings with the matches printed.

**How to use this notebook:** run cells top to bottom; later sections reuse the `Graph` class. Companion reading: Chapters 37 to 41.

## 1. Two representations, one trade-off

There are two serious ways to store a graph.

An **adjacency matrix** is an $n \times n$ grid where cell $(u, v)$ says whether the edge exists. Checking one edge is a single index, which is as fast as anything gets — but the storage is $\Theta(V^2)$ whether or not the edges exist, and listing one vertex's neighbours means scanning a whole row.

An **adjacency list** stores, per vertex, only its actual neighbours. Storage is $\Theta(V + E)$, listing neighbours is $\Theta(\deg(u))$, and if you use a dict per vertex then edge lookup is $O(1)$ too.

The question is which regime you are in. A graph is **dense** when $E$ is close to $V^2$ and **sparse** when $E$ is closer to $V$. Real graphs are overwhelmingly sparse, and the cell below prices that.

In [ ]:
class MatrixGraph:
    """Adjacency matrix: a V x V grid of 0/1."""

    def __init__(self, n, directed=False):
        self.n = n
        self.directed = directed
        self.m = [[0] * n for _ in range(n)]

    def add_edge(self, u, v):
        self.m[u][v] = 1
        if not self.directed:
            self.m[v][u] = 1

    def has_edge(self, u, v):
        return self.m[u][v] == 1              # O(1): two index operations

    def neighbours(self, u):
        return [v for v in range(self.n) if self.m[u][v]]     # O(V): whole row

class ListGraph:
    """Adjacency list: per vertex, only the neighbours that exist."""

    def __init__(self, n, directed=False):
        self.n = n
        self.directed = directed
        self.adj = {u: set() for u in range(n)}

    def add_edge(self, u, v):
        self.adj[u].add(v)
        if not self.directed:
            self.adj[v].add(u)

    def has_edge(self, u, v):
        return v in self.adj[u]               # O(1) expected: a set lookup

    def neighbours(self, u):
        return sorted(self.adj[u])            # O(deg(u))

import random

def compare(n, degree, seed=37):
    """Build the same random graph both ways and count what each one stores."""
    rng = random.Random(seed)
    edges = set()
    while len(edges) < n * degree // 2:
        u, v = rng.randrange(n), rng.randrange(n)
        if u != v:
            edges.add((min(u, v), max(u, v)))
    mg, lg = MatrixGraph(n), ListGraph(n)
    for u, v in edges:
        mg.add_edge(u, v)
        lg.add_edge(u, v)
    matrix_cells = n * n
    list_cells = 2 * len(edges)
    # Cost of one full traversal: every vertex asks for its neighbours.
    matrix_scan = sum(len(range(n)) for _ in range(n))          # V rows of V
    list_scan = sum(len(lg.adj[u]) for u in range(n))           # sum of degrees
    return len(edges), matrix_cells, list_cells, matrix_scan, list_scan

print(f"{'V':>6}{'avg deg':>9}{'E':>8}{'matrix cells':>14}{'list cells':>12}"
      f"{'ratio':>8}{'scan: matrix':>14}{'list':>8}")
for n, deg in [(50, 4), (200, 4), (1000, 4), (200, 20), (200, 190)]:
    e, mc, lc, ms, ls = compare(n, deg)
    print(f"{n:>6}{deg:>9}{e:>8}{mc:>14,}{lc:>12,}{mc / max(lc, 1):>7.0f}x"
          f"{ms:>14,}{ls:>8,}")

# Both answer the same questions, so the choice is purely about cost.
mg, lg = MatrixGraph(5), ListGraph(5)
for u, v in [(0, 1), (0, 2), (1, 3), (3, 4)]:
    mg.add_edge(u, v)
    lg.add_edge(u, v)
print("\nsame answers:",
      all(mg.has_edge(u, v) == lg.has_edge(u, v)
          for u in range(5) for v in range(5)),
      " neighbours of 0:", mg.neighbours(0), lg.neighbours(0))

Read the last two rows against the first three. A 200-vertex graph with average degree 4 stores 800 numbers in a list and 40,000 in a matrix — a fiftyfold waste — and the gap widens as $V$ grows, because the matrix grows quadratically while the list grows linearly: at 1000 vertices the same average degree makes it 250-fold. Only in the last row, at average degree 190 out of a possible 199, do the two become comparable, and there the matrix's constant-time simplicity wins.

The `scan` columns are the same story for *time*. One full traversal reads $V^2$ cells from a matrix regardless of edge count, and $2E$ from a list. That is precisely why every algorithm below is quoted as $O(V + E)$: it assumes an adjacency list. Run BFS on a matrix and you get $O(V^2)$ for free, with no warning.

Rule of thumb: use an adjacency list unless you know your graph is dense, or you need matrix arithmetic (which is a real reason — Floyd-Warshall and spectral methods want the matrix).

## 2. BFS and DFS: the same code, one line apart

Everything from here uses one small class: a dict of dicts mapping each vertex to `{neighbour: weight}`. Weights default to 1, so the same class serves unweighted algorithms with no second code path.

In [ ]:
class Graph:
    """Adjacency-map graph: {vertex: {neighbour: weight}}."""

    def __init__(self, directed=False):
        self.adj = {}
        self.directed = directed

    def add_vertex(self, v):
        self.adj.setdefault(v, {})
        return self

    def add_edge(self, u, v, weight=1):
        self.add_vertex(u)
        self.add_vertex(v)
        self.adj[u][v] = weight
        if not self.directed:
            self.adj[v][u] = weight
        return self

    def neighbours(self, u):
        return self.adj[u].keys()

    def weight(self, u, v):
        return self.adj[u][v]

    def has_edge(self, u, v):
        return v in self.adj.get(u, {})       # O(1): a dict lookup

    def vertices(self):
        return self.adj.keys()

    def edges(self):
        """Each undirected edge yielded once, as (u, v, weight)."""
        seen = set()
        for u in self.adj:
            for v, w in self.adj[u].items():
                if self.directed or (v, u) not in seen:
                    seen.add((u, v))
                    yield u, v, w

    def __len__(self):
        return len(self.adj)

g = (Graph()
     .add_edge("A", "B").add_edge("A", "C")
     .add_edge("B", "D").add_edge("C", "D")
     .add_edge("D", "E").add_edge("E", "F")
     .add_edge("C", "G"))
print(len(g), "vertices,", len(list(g.edges())), "edges")
for v in sorted(g.vertices()):
    print(f"  {v}: {sorted(g.neighbours(v))}")

Now the two traversals. **Breadth-first search** visits everything one edge away, then everything two edges away, and so on — rings around the start. **Depth-first search** dives as far as it can and backtracks only when stuck.

The structural difference between them is *one data structure*. BFS takes the **oldest** pending vertex (a queue); DFS takes the **newest** (a stack). Everything else — the visited set, the loop, the marking — is identical, and the cell below proves it by writing the shared version once.

In [ ]:
from collections import deque

def traverse(graph, start, mode):
    """BFS if mode == 'bfs', DFS if mode == 'dfs'. One line differs."""
    frontier = deque([start])
    visited = {start}
    order = []
    while frontier:
        v = frontier.popleft() if mode == "bfs" else frontier.pop()
        order.append(v)
        for nb in sorted(graph.neighbours(v)):
            if nb not in visited:
                visited.add(nb)               # mark on ENQUEUE, not on visit
                frontier.append(nb)
    return order

def bfs_levels(graph, start):
    """How many edges from the start is each vertex? BFS answers this for free."""
    level = {start: 0}
    frontier = deque([start])
    while frontier:
        v = frontier.popleft()
        for nb in sorted(graph.neighbours(v)):
            if nb not in level:
                level[nb] = level[v] + 1
                frontier.append(nb)
    return level

print("BFS from A:", traverse(g, "A", "bfs"))
print("DFS from A:", traverse(g, "A", "dfs"))
print("\nBFS levels (edges from A):", bfs_levels(g, "A"))
print("shortest unweighted distance A -> F:", bfs_levels(g, "A")["F"])

# The visited set is not optional: this graph has a cycle A-B-D-C-A.
print("\nis there a cycle through A, B, D, C?",
      all(g.has_edge(u, v) for u, v in [("A", "B"), ("B", "D"),
                                        ("D", "C"), ("C", "A")]))
print("both traversals still terminate and visit every vertex exactly once:",
      len(traverse(g, "A", "bfs")) == len(g) == len(set(traverse(g, "A", "dfs"))))

Two different orders from the same graph and the same code. BFS reaches `B` and `C` before touching `D`, because they are one edge out; DFS commits to a path and follows it to the end.

The `bfs_levels` output is the reason BFS matters beyond bookkeeping: on an **unweighted** graph, BFS finds shortest paths, because it cannot reach a vertex in three edges before it has finished checking every two-edge route. That guarantee is exactly what breaks when edges get weights, which is section 4.

Two details that are bugs waiting to happen. First, mark a vertex visited **when you enqueue it**, not when you pop it — otherwise a vertex with two unvisited neighbours pointing at it gets added twice, and on a large graph the frontier explodes. Second, the visited set is not an optimisation, it is what makes the algorithm terminate: our graph contains the cycle A–B–D–C–A, and without the set both traversals would walk it forever.

## 3. Topological sort: an order that respects every arrow

On a **directed acyclic graph** (DAG) a natural question is "what order can I do these in, if every arrow means *must come first*?" That is a **topological sort**, and it is what `make`, package managers, build systems, and course planners all compute.

**Kahn's algorithm** is BFS with a twist: repeatedly take any vertex with **in-degree 0** (nothing left blocking it), emit it, and decrement its neighbours' in-degrees. If you run out of zero-in-degree vertices before emitting everything, the remainder contains a cycle — so the same algorithm is also a cycle detector.

In [ ]:
import heapq

PREREQS = [
    ("intro", "data-structures"), ("intro", "discrete-math"),
    ("data-structures", "algorithms"), ("discrete-math", "algorithms"),
    ("data-structures", "databases"), ("algorithms", "compilers"),
    ("architecture", "operating-systems"), ("intro", "architecture"),
    ("operating-systems", "networks"), ("algorithms", "machine-learning"),
    ("discrete-math", "machine-learning"),
]
courses = Graph(directed=True)
for a, b in PREREQS:
    courses.add_edge(a, b)

def kahn(graph):
    """Topological order, alphabetical among ties. Returns (order, ok)."""
    in_degree = {v: 0 for v in graph.vertices()}
    for u in graph.vertices():
        for v in graph.neighbours(u):
            in_degree[v] += 1
    ready = [v for v, d in in_degree.items() if d == 0]
    heapq.heapify(ready)                     # a heap only to make ties tidy
    order = []
    while ready:
        v = heapq.heappop(ready)
        order.append(v)
        for nb in sorted(graph.neighbours(v)):
            in_degree[nb] -= 1
            if in_degree[nb] == 0:
                heapq.heappush(ready, nb)
    return order, len(order) == len(graph)

order, ok = kahn(courses)
print("valid topological order:", ok)
for i, course in enumerate(order, start=1):
    blockers = sorted(u for u in courses.vertices()
                      if courses.has_edge(u, course))
    print(f"  {i:>2}. {course:<20} after {blockers or ['-']}")

# The order is only valid if every edge points forwards in it:
position = {v: i for i, v in enumerate(order)}
print("\nevery prerequisite comes strictly earlier:",
      all(position[u] < position[v] for u, v in PREREQS))

# Add one edge that closes a cycle, and Kahn refuses.
courses.add_edge("machine-learning", "intro")
bad_order, bad_ok = kahn(courses)
print("\nafter adding machine-learning -> intro:")
print("  valid order:", bad_ok, " emitted", len(bad_order), "of", len(courses))
print("  stuck inside the cycle:",
      sorted(set(courses.vertices()) - set(bad_order)))

The verification line is the point. A topological order is not "an order that looks plausible" — it is an order in which **every** edge points forwards, and that is one line to check, so check it.

When the cycle is added, Kahn emits **nothing at all**, because every vertex now has in-degree above zero: `intro` needs `machine-learning`, which needs `algorithms`, which needs `data-structures`, which needs `intro`. The reported set is all ten vertices, which is worth reading carefully — `networks` and `databases` are not in the cycle, they are merely downstream of it, and nothing can be scheduled while their ancestors are stuck.

Notice how much more useful that is than a crash. A build system that says "these targets could not be ordered" has told you where to look; one that recurses until the stack blows has not. (Narrowing the report to the cycle itself means finding strongly connected components — a DFS refinement, and a natural next step from here.)

Kahn runs in $O(V + E)$: each vertex is pushed and popped once, each edge decrements one counter once.

## 4. Dijkstra: BFS with a priority queue

Give the edges weights and BFS's guarantee evaporates: the route with the fewest edges need not be the cheapest. **Dijkstra's algorithm** replaces BFS's queue with a **min-heap** keyed on tentative distance, so instead of "the vertex found earliest" it always expands "the unsettled vertex with the smallest known distance".

The invariant that makes it correct: when a vertex is popped with distance $d$, no cheaper route to it can exist, because any other route would have to pass through an unsettled vertex whose distance is already $\ge d$ — *and the rest of that route only adds length*. Remember that last clause; section 5 is about what happens when it is false.

The implementation prints the frontier after every settle, so you can watch the algorithm think.

In [ ]:
INF = float("inf")

roads = (Graph()
         .add_edge("Ashby", "Brook", 4).add_edge("Ashby", "Crest", 2)
         .add_edge("Brook", "Crest", 1).add_edge("Brook", "Dune", 5)
         .add_edge("Crest", "Dune", 8).add_edge("Crest", "Elm", 10)
         .add_edge("Dune", "Elm", 2).add_edge("Dune", "Ford", 6)
         .add_edge("Elm", "Ford", 3))

def dijkstra(graph, src, trace=False):
    dist = {v: INF for v in graph.vertices()}
    parent = {v: None for v in graph.vertices()}
    dist[src] = 0
    settled = set()
    heap = [(0, src)]                        # (tentative distance, vertex)
    step = 0
    if trace:
        print(f"{'step':>4} {'settle':>7} {'dist':>5}  "
              f"{'heap contents (the frontier)':<36} improved")
    while heap:
        d, u = heapq.heappop(heap)
        if u in settled:                     # a stale copy; see below
            continue
        settled.add(u)
        step += 1
        improved = []
        for v in sorted(graph.neighbours(u)):
            if v in settled:
                continue
            w = graph.weight(u, v)
            if d + w < dist[v]:
                dist[v] = d + w
                parent[v] = u
                heapq.heappush(heap, (d + w, v))
                improved.append(f"{v}={d + w}")
        if trace:
            front = ", ".join(f"{v}:{x}" for x, v in sorted(heap))
            print(f"{step:>4} {u:>7} {d:>5}  {front:<36} {' '.join(improved)}")
    return dist, parent

def path_to(parent, target):
    path = []
    while target is not None:
        path.append(target)
        target = parent[target]
    return path[::-1]

dist, parent = dijkstra(roads, "Ashby", trace=True)
print("\nfinal distances from Ashby:")
for town in sorted(dist, key=dist.get):
    print(f"  {town:<6} {dist[town]:>3} km   {' -> '.join(path_to(parent, town))}")

Three things in that trace are worth your attention.

**Step 2 improves a route that already existed.** Ashby–Brook is a direct 4 km road, but Ashby → Crest → Brook costs $2 + 1 = 3$. The moment Crest is settled, `Brook=3` replaces the direct road. A greedy algorithm that committed to the first route it found would have kept the 4.

**The cheapest route has the most hops.** Ashby to Ford is five edges long, and every shorter-hop alternative is no better or much worse. "Fewest edges" and "least weight" are different questions, which is the whole reason Dijkstra exists.

**The heap fills with stale entries.** At step 2 the frontier holds both `Brook:3` and `Brook:4` — two copies of the same vertex at different tentative distances. Textbook Dijkstra says "decrease the key of $v$ in the priority queue", but `heapq` has no decrease-key, so we push a second entry and skip any vertex already settled. That is **lazy deletion**: the heap holds up to $E$ entries instead of $V$, costing $O(E \log E)$ instead of $O(E \log V)$ — the same complexity class, in exchange for ten lines you do not have to write.

Path reconstruction needs one array. Every time we improve `dist[v]` we also record *who* improved it, and walking `parent` backwards from any vertex gives the route. Storing the whole path per vertex instead would cost $O(V^2)$ space for no benefit.

## 5. Where Dijkstra is simply wrong

Now the clause we were told to remember: *the rest of the route only adds length*. Negative edge weights make that false, and when it is false the greedy commitment — settle a vertex, never reconsider it — collapses.

Negative weights are not a contrivance. A freight network where consolidating a load earns a rebate, a currency-exchange graph where a sequence of trades yields a profit, a game where crossing a tile restores health: all have edges that reduce the running total.

Read this graph before running it. Depot → Port directly costs 2. Depot → Hub → Port costs $3 + (-2) = 1$, which is cheaper. Dijkstra settles Port at 2 before it has ever looked at Hub, and never reconsiders.

In [ ]:
freight = (Graph(directed=True)
           .add_edge("Depot", "Port", 2).add_edge("Depot", "Hub", 3)
           .add_edge("Hub", "Port", -2)        # a rebate for consolidating
           .add_edge("Port", "Yard", 4))
freight.add_vertex("Yard")

truth = {"Depot": 0, "Hub": 3, "Port": 1, "Yard": 5}
got, _ = dijkstra(freight, "Depot")
print(f"{'vertex':>7}{'Dijkstra':>10}{'truth':>7}   verdict")
for v in ("Depot", "Hub", "Port", "Yard"):
    mark = "ok" if got[v] == truth[v] else f"WRONG (off by {got[v] - truth[v]})"
    print(f"{v:>7}{got[v]:>10}{truth[v]:>7}   {mark}")
print("\nNote the error propagates: Yard is wrong only because Port was.")

Two wrong answers, no exception, no warning. This is the failure mode to memorise, because it looks exactly like a correct run.

**Bellman-Ford** gives up the greedy shortcut and does something almost embarrassingly simple: relax *every* edge, $V - 1$ times. After pass $k$, every shortest path using at most $k$ edges is correct; since no shortest path in a graph without negative cycles uses more than $V - 1$ edges, $V - 1$ passes suffice. The cost is $O(V \cdot E)$ instead of $O(E \log V)$ — genuinely slower, and right.

And one extra pass is a **negative-cycle detector**: if anything still improves after $V - 1$ passes, some cycle has negative total weight, and "shortest path" has no answer at all — you can go round it forever getting cheaper.

In [ ]:
def bellman_ford(vertices, edges, src, trace=False):
    dist = {v: INF for v in vertices}
    parent = {v: None for v in vertices}
    dist[src] = 0
    for i in range(len(vertices) - 1):
        changed = False
        for u, v, w in edges:
            if dist[u] != INF and dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                parent[v] = u
                changed = True
        if trace:
            shown = {k: (v if v != INF else "inf") for k, v in dist.items()}
            print(f"  after pass {i + 1}: {shown}")
        if not changed:
            if trace:
                print("  nothing changed: stopping early")
            break
    for u, v, w in edges:                    # one extra pass = the detector
        if dist[u] != INF and dist[u] + w < dist[v]:
            raise ValueError(f"negative cycle reachable via edge {u} -> {v}")
    return dist, parent

vertices = ["Depot", "Hub", "Port", "Yard"]
edges = [("Depot", "Port", 2), ("Depot", "Hub", 3),
         ("Hub", "Port", -2), ("Port", "Yard", 4)]

dist_bf, parent_bf = bellman_ford(vertices, edges, "Depot", trace=True)
print("\nBellman-Ford matches the truth:", dist_bf == truth)
print("cheapest Depot -> Port:", " -> ".join(path_to(parent_bf, "Port")),
      "=", dist_bf["Port"])
print("cheapest Depot -> Yard:", " -> ".join(path_to(parent_bf, "Yard")),
      "=", dist_bf["Yard"])

# Now make the cycle negative and watch the detector fire.
cyclic = edges + [("Yard", "Depot", -8)]
print("\nadding Yard -> Depot with weight -8 (total cycle weight "
      f"{2 + 4 - 8}):")
try:
    bellman_ford(vertices, cyclic, "Depot")
except ValueError as exc:
    print("  ValueError:", exc)

print(f"\n{'algorithm':<16}{'handles negative w':>20}{'cost':>16}")
print(f"{'Dijkstra':<16}{'no':>20}{'O(E log V)':>16}")
print(f"{'Bellman-Ford':<16}{'yes (+ detects cycles)':>20}{'O(V*E)':>16}")

The trace shows the "at most $k$ edges" property directly: pass 1 already has every distance right, because this graph's longest shortest path is short, and the early-exit check then stops the loop instead of grinding through the remaining passes. That early exit is free and worth always including — on many real graphs Bellman-Ford converges in two or three passes.

The negative-cycle case is not a hypothetical. Its most famous application is *arbitrage detection*: take exchange rates, use $-\log(\text{rate})$ as the edge weight, and a negative cycle is a sequence of trades that returns more money than it started with.

The decision rule is simple. All weights non-negative? Dijkstra. Any negative weight? Bellman-Ford. Need every pair of distances at once on a small dense graph? Floyd-Warshall, which is a triple loop and three lines long. Have a geometric hint about where the target is? A\*, which is Dijkstra with the priority `dist + heuristic`.

## 6. Minimum spanning trees: Kruskal and union-find

A different question: connect every vertex as cheaply as possible. The answer is a **minimum spanning tree** — $V - 1$ edges, no cycles, minimum total weight. (It must be a tree: any cycle contains an edge you could delete while staying connected, so a cheapest connected subgraph is never cyclic.)

**Kruskal's algorithm** is greed at its purest: sort all edges by weight, and accept each one unless it would close a cycle. Which needs one capability — "are these two vertices already connected?" — asked $E$ times. That is **union-find** (a disjoint-set forest): every set has a representative, `find` walks to it, and `union` links two representatives. With *path compression* (point every node you passed directly at the root) and *union by rank* (hang the shorter tree under the taller), both operations are effectively $O(1)$.

In [ ]:
class UnionFind:
    def __init__(self, items):
        self.parent = {x: x for x in items}
        self.rank = {x: 0 for x in items}

    def find(self, x):
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:        # path compression, iteratively
            self.parent[x], x = root, self.parent[x]
        return root

    def union(self, a, b):
        """Join two sets. Returns False if they were already the same set."""
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra                  # union by rank: shorter under taller
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1
        return True

uf = UnionFind("ABCDE")
print("A and C connected?", uf.find("A") == uf.find("C"))
for a, b in [("A", "B"), ("C", "D"), ("B", "C")]:
    print(f"union({a}, {b}) ->", uf.union(a, b))
print("union(A, D) ->", uf.union("A", "D"), " (already connected: no-op)")
print("A and D connected?", uf.find("A") == uf.find("D"))
print("E is still alone:", uf.find("E") == "E")

Now Kruskal, printing *why* each edge was accepted or rejected.

In [ ]:
EDGES = [("A", "B", 7), ("A", "D", 5), ("B", "C", 8), ("B", "D", 9),
         ("B", "E", 7), ("C", "E", 5), ("D", "E", 15), ("D", "F", 6),
         ("E", "F", 8), ("E", "G", 9), ("F", "G", 11)]
VERTICES = sorted({v for u, w, _ in EDGES for v in (u, w)})

def kruskal(vertices, edges, trace=False):
    uf = UnionFind(vertices)
    tree = []
    if trace:
        print(f"{'edge':>6}{'w':>4}   decision")
    for u, v, w in sorted(edges, key=lambda e: e[2]):    # the sort is the cost
        if uf.union(u, v):
            tree.append((u, v, w))
            if trace:
                print(f"{u + '-' + v:>6}{w:>4}   accept - joins two components")
        elif trace:
            print(f"{u + '-' + v:>6}{w:>4}   REJECT - both already in "
                  f"component {uf.find(u)!r}")
        if len(tree) == len(vertices) - 1:
            break                                        # early exit: done
    return tree

tree = kruskal(VERTICES, EDGES, trace=True)
print("\nMST edges:", tree)
print("edges:", len(tree), "= V - 1 =", len(VERTICES) - 1)
print("total weight:", sum(w for *_, w in tree))

# Verify it really is a spanning tree: connected, and acyclic.
check = UnionFind(VERTICES)
acyclic = all(check.union(u, v) for u, v, _ in tree)
roots = {check.find(v) for v in VERTICES}
print("acyclic:", acyclic, " connected:", len(roots) == 1)

Kruskal's shape is completely different from Prim's, which grows one blob outward from a start vertex. Kruskal happily accepts the two weight-5 edges first even though they are in *opposite corners* of the graph — for most of the run it maintains a whole forest of disconnected fragments, and only at the end do they merge into one tree.

The rejections are the interesting lines: each names the component both endpoints already share, which is exactly the cycle that would have been created. And note the early exit — as soon as $V - 1$ edges are accepted the remaining edges cannot help, so the loop stops before examining the heaviest ones.

Cost: $O(E \log E)$, and that is entirely the sort. The union-find work is effectively linear. So on a sparse graph Kruskal is essentially "sort the edges", which is why it is the easier of the two MST algorithms to remember.

## 7. Sorting without comparing

Chapter 22 proved that any sort which learns about the data *only through comparisons* needs $\Omega(n \log n)$ comparisons: $n!$ possible orders, one bit per comparison, $\log_2 (n!) \approx n \log_2 n$. The bound is airtight — and it has a loophole. If you can look *at* the key rather than only comparing it, you are not in the model, and the bound does not apply.

**Counting sort** is the purest form. If keys are integers in $[0, k)$, tally how many of each, turn the tallies into starting positions with a running total, then place each item directly at its address. Three passes, no comparison anywhere.

In [ ]:
records = [("Ana", 2), ("Bo", 1), ("Cy", 2), ("Di", 0), ("Eve", 1), ("Fay", 2)]

def counting_sort(items, k, show=False):
    """Sort (name, key) pairs by key, where 0 <= key < k. Stable."""
    n = len(items)

    count = [0] * k                          # pass 1: tally
    for _, key in items:
        count[key] += 1
    if show:
        print("  counts        :", count)

    for v in range(1, k):                    # pass 2: running total
        count[v] += count[v - 1]
    if show:
        print("  prefix sums   :", count, " <- count[v] = where group v ENDS")

    out = [None] * n                         # pass 3: place, backwards
    for i in range(n - 1, -1, -1):
        name, key = items[i]
        count[key] -= 1                      # the slot this item claims
        out[count[key]] = (name, key)
        if show:
            print(f"  place {name:>4}/{key} at index {count[key]}   {out}")
    return out

print("input :", records)
result = counting_sort(records, 3, show=True)
print("output:", result)
print("keys sorted:", [r[1] for r in result] == sorted(r[1] for r in records))
print("stable (ties keep input order):",
      [n for n, k in result if k == 2] == [n for n, k in records if k == 2])

Read the prefix sums: `count[v]` becomes the index one *past* where group $v$ ends. So walking the input **backwards** and pre-decrementing gives each item the last free slot of its group — which means items that were later in the input land later in the output. That is **stability**, and it is not decoration here: it is the property the next algorithm is built on.

Cost: $O(n + k)$. The $k$ is the catch. Counting-sort a million 32-bit integers and you allocate a four-billion-entry tally array; the algorithm is linear in $n$ and useless. It works when the key range is small — exam scores, ages, bytes, priority levels.

**Radix sort** removes that limit by sorting one *digit* at a time, least significant first, using a stable counting sort per digit. It looks like it must be wrong — surely the most significant digit matters most? — and stability is exactly why it is right: when the tens pass runs, the array is already sorted by ones, and a stable sort by tens preserves that order within each group of equal tens digits.

In [ ]:
def radix_sort(values, base=10, show=False):
    """LSD radix sort using a stable counting sort per digit."""
    a = list(values)
    if not a:
        return a
    largest = max(a)
    place, pass_no = 1, 1
    while largest // place > 0:
        count = [0] * base                               # counting sort, one digit
        for x in a:
            count[(x // place) % base] += 1
        for d in range(1, base):
            count[d] += count[d - 1]
        out = [0] * len(a)
        for i in range(len(a) - 1, -1, -1):               # backwards: STABLE
            digit = (a[i] // place) % base
            count[digit] -= 1
            out[count[digit]] = a[i]
        a = out
        if show:
            print(f"  pass {pass_no} (digit worth {place:>3}): {a}")
        place *= base
        pass_no += 1
    return a

data = [170, 45, 75, 90, 802, 24, 2, 66]
print("input :", data)
result = radix_sort(data, base=10, show=True)
print("output:", result)
print("matches sorted():", result == sorted(data))

# Break the stability and the sort breaks with it.
def radix_sort_unstable(values, base=10):
    a = list(values)
    place = 1
    while max(a) // place > 0:
        buckets = [[] for _ in range(base)]
        for x in a:
            buckets[(x // place) % base].append(x)
        a = [x for b in buckets for x in reversed(b)]     # <- the only change
        place *= base
    return a

print("\nwith one pass made unstable:", radix_sort_unstable(data))
print("still correct?", radix_sort_unstable(data) == sorted(data))

Watch the passes. After pass 1 the list looks like noise. After pass 2 it looks *worse* — 802 near the front, 170 in the middle. After pass 3 it is perfectly sorted, and not one comparison between two values was ever made.

Look at pass 2 to catch stability working: `2` and `802` both have tens digit 0, and `2` came after `802` in the pass-1 output, so it stays after `802`. The hundreds pass then separates them correctly. Reverse each bucket instead — the `radix_sort_unstable` variant — and the whole thing collapses, because each pass now destroys the ordering the previous pass established.

Cost: $O(d(n + b))$ for $d$ digits in base $b$. For 32-bit integers in base 256 that is $4(n + 256)$ — linear in $n$ with a constant around 4. Choosing the base is a real trade: bigger base, fewer passes, bigger tally array.

When *not* to use it: radix sort needs fixed-width keys, several full passes over the data (bad for cache compared with an in-place quicksort), and extra memory. `sorted()` is a highly tuned adaptive mergesort that exploits existing runs in the data; beating it requires the key structure to genuinely be on your side.

### Micro-exercise: counting sort on characters

Write `sort_letters(text)` that sorts the lowercase letters of a string using `counting_sort`'s idea directly on character codes — tally 26 counters, then emit each letter `count[i]` times. No comparison, no `sorted()`.

In [ ]:
def sort_letters(text):
    count = [0] * 26
    # your code here: tally ord(ch) - ord('a') for the lowercase letters,
    # then build the output string from the counts
    return ""

# Uncomment to test:
# assert sort_letters("banana") == "aaabnn"
# print(sort_letters("the quick brown fox"))

## 8. A regex lab

A **regular expression** is a small language for describing sets of strings, and `re` is in the standard library — so unlike most of the toolchain, this part is fully runnable. Every pattern below is executed against real strings with the matches printed, because a regex you have not run is a regex you do not know the behaviour of.

The single most useful feature for real work is the **named group**: `(?P<name>...)` captures a piece and labels it, and `m.groupdict()` hands you a dictionary. Combined with `re.VERBOSE`, which lets you break a pattern across lines and comment it, a parser stops being a mystery string.

In [ ]:
import re

# Write the log file first: Pyodide has an in-memory filesystem.
with open("access.log", "w", encoding="utf-8") as f:
    f.write("""\
203.0.113.7 - - [05/Feb/2024:10:12:44] "GET /index.html HTTP/1.1" 200 5120
198.51.100.22 - - [05/Feb/2024:10:12:45] "GET /admin HTTP/1.1" 403 210
203.0.113.7 - - [05/Feb/2024:10:12:46] "POST /api/login HTTP/1.1" 200 88
-- corrupt line, no fields at all --
198.51.100.22 - - [05/Feb/2024:10:13:01] "GET /admin HTTP/1.1" 403 210
192.0.2.66 - - [05/Feb/2024:10:13:09] "GET /cart HTTP/1.1" 500 0
203.0.113.7 - - [05/Feb/2024:10:14:22] "GET /index.html HTTP/1.1" 200 5120
198.51.100.22 - - [05/Feb/2024:10:15:00] "DELETE /api/user/7 HTTP/1.1" 500 0
""")

LOG = re.compile(r"""
    ^(?P<ip>\d{1,3}(?:\.\d{1,3}){3})\s+\S+\s+\S+\s+   # dotted-quad address
    \[(?P<time>[^\]]+)\]\s+                           # [timestamp]
    "(?P<method>[A-Z]+)\s(?P<path>\S+)[^"]*"\s+       # "METHOD /path HTTP/x"
    (?P<status>\d{3})\s+(?P<size>\d+)$                # status and byte count
""", re.VERBOSE)

records, skipped = [], []
with open("access.log", encoding="utf-8") as f:
    for line in f:
        m = LOG.match(line.strip())
        if m is None:
            skipped.append(line.strip())          # collected, never ignored
            continue
        record = m.groupdict()
        record["status"] = int(record["status"])   # convert at the boundary
        record["size"] = int(record["size"])
        records.append(record)

print(f"parsed {len(records)} records, skipped {len(skipped)}")
print("skipped:", skipped)
print("\nfirst record:", records[0])

by_status, by_path, offenders = {}, {}, {}
for r in records:
    by_status[r["status"]] = by_status.get(r["status"], 0) + 1
    by_path[r["path"]] = by_path.get(r["path"], 0) + 1
    if r["status"] >= 400:
        offenders[r["ip"]] = offenders.get(r["ip"], 0) + 1

print("\nstatus counts:", dict(sorted(by_status.items())))
print("busiest paths:", sorted(by_path.items(), key=lambda kv: (-kv[1], kv[0]))[:3])
failures = sum(1 for r in records if r["status"] >= 400)
print(f"failure rate : {failures}/{len(records)} = "
      f"{100 * failures / len(records):.0f}%")
print("failing IPs  :", sorted(offenders.items(), key=lambda kv: -kv[1]))
print("bytes served :", sum(r["size"] for r in records))

Three habits from that code are worth copying. The pattern is a **named constant in verbose mode with comments**, not an inline mystery string. Non-matching lines are **collected, not silently dropped** — a parser that ignores what it does not understand will hide the day the log format changes. And the conversion to `int` happens at the boundary, right after matching, so the rest of the program works with numbers instead of strings.

Now the trap that catches everybody: **greedy versus lazy**. `*` and `+` are greedy — they match as much as possible and then give characters back only if the rest of the pattern fails. Add `?` and they become lazy, matching as little as possible.

In [ ]:
html = '<b>bold</b> and <i>italic</i>'
print("greedy  <.+> :", re.findall(r"<.+>", html))
print("lazy    <.+?>:", re.findall(r"<.+?>", html))
print("better  <[^>]+>:", re.findall(r"<[^>]+>", html))

quoted = 'she said "one" then "two"'
print('\ngreedy  ".+" :', re.findall(r'".+"', quoted))
print('lazy    ".+?":', re.findall(r'".+?"', quoted))

# The dot does not match a newline unless you ask (re.DOTALL):
two_lines = "start\nend"
print("\n'.+' across a newline:", re.findall(r".+", two_lines))
print("'.+' with DOTALL      :", re.findall(r".+", two_lines, re.DOTALL))

# Anchors and word boundaries change what "match" even means:
print("\n'cat' in 'concatenate' :", bool(re.search(r"cat", "concatenate")))
print(r"'\bcat\b' in 'concatenate':", bool(re.search(r"\bcat\b", "concatenate")))
print(r"'\bcat\b' in 'the cat sat':", bool(re.search(r"\bcat\b", "the cat sat")))

`<.+>` swallows the entire string in one match, because `.` happily eats `>` and the greedy `+` only backs off far enough to find the *last* `>`. `<.+?>` stops at the first `>` and returns the four tags. And `<[^>]+>` — a **negated character class** — is better than either, because it cannot cross a `>` at all: no backtracking, no ambiguity, and it stays correct if you later add more to the pattern.

That last point is also the safety point. Nested quantifiers over overlapping alternatives, like `(a+)+b`, can take exponential time on a non-matching string — **catastrophic backtracking**, and it is a real denial-of-service vector. The defence is not cleverness; it is preferring precise character classes over `.*`, and avoiding a quantifier applied to something that is already quantified.

The third piece of the toolkit is `sub` with a **function** as the replacement. Anything you can compute in Python can then rewrite the match, which turns regex from a search tool into a transformation tool.

In [ ]:
# 1. Reformat dates using named groups in the replacement string.
NAMED = r"(?P<y>\d{4})-(?P<m>\d{2})-(?P<d>\d{2})"
print(re.sub(NAMED, r"\g<d>/\g<m>/\g<y>", "due 2024-02-05, shipped 2024-03-11"))

# 2. A function replacement: keep the last four digits, mask the rest.
CARD = re.compile(r"\b\d(?:[ -]?\d){12,18}\b")      # 13-19 digits, spaced or hyphenated

def mask(m):
    digits = re.sub(r"\D", "", m.group())
    return "*" * (len(digits) - 4) + digits[-4:]

print(re.sub(CARD, mask, "pay 4111 1111 1111 1111 or 5500-0000-0000-0004 today"))

# 3. A function replacement that decides per match.
def shout_long_words(m):
    word = m.group()
    return word.upper() if len(word) > 5 else word

print(re.sub(r"[A-Za-z]+", shout_long_words,
             "the quick brown foxes jumped over lazy dogs"))

# 4. Counting and locating every match, with positions.
TIME = re.compile(r"\b(?P<h>\d{1,2}):(?P<min>\d{2})\b")
line = "standup 09:30, review 14:05, retro 16:45"
for m in TIME.finditer(line):
    print(f"  {m.group():>5} at chars {m.start()}-{m.end()}  "
          f"hour={m['h']:>2} minute={m['min']}")
print("  as tuples:", TIME.findall(line))

Two details in that output repay attention. In `re.sub`, `\g<d>` refers to a named group in the *replacement* — a plain `\1` works too, but the name survives you reordering the pattern. And `findall` returns **tuples of groups** when the pattern has groups, not the whole match: that surprise is why `finditer` is usually the better habit, since a match object gives you the text, the groups, *and* the positions.

The masking example is also a small lesson in scope. `mask` strips non-digits itself rather than trying to express "digits possibly separated by spaces or hyphens, count them, keep four" inside the pattern. Let the regex find the *region*, then let Python do the logic. Patterns that try to compute become unreadable long before they become wrong.

## 9. Generators as a Unix pipeline

The last tool is a shape rather than a library. A shell pipeline

```console
cat access.log | grep ERROR | cut -d' ' -f1 | sort | uniq -c | sort -rn | head -3
```

is a chain of small programs, each reading a stream and writing a stream, none of which holds the whole file. Python generators are exactly that: a function with `yield` produces values **on demand**, so composing generators composes streams and memory stays constant no matter how large the input.

Each stage below is a generator except the two that genuinely cannot be — `sort` must see everything before it can emit anything, which is why `sort` is the stage that blocks in a real pipeline too.

In [ ]:
LOG_TEXT = """\
2024-02-05 10:12:44 INFO  ingest  batch accepted
2024-02-05 10:12:45 ERROR ingest  batch too large
2024-02-05 10:12:46 INFO  auth    login ok
2024-02-05 10:13:01 ERROR ingest  batch too large
2024-02-05 10:13:09 ERROR auth    token expired
2024-02-05 10:14:22 WARN  ingest  retrying
2024-02-05 10:15:00 ERROR ingest  batch too large
2024-02-05 10:15:30 ERROR report  timeout
"""

def cat(text):                       # cat access.log
    for line in text.splitlines():
        yield line

def grep(pattern, lines):            # grep ERROR
    for line in lines:
        if pattern in line:
            yield line

def cut(lines, field, delim=None):   # cut -f<field>
    for line in lines:
        parts = line.split(delim)
        if field < len(parts):
            yield parts[field]

def sort_stage(items):               # sort   (BLOCKING: consumes everything)
    yield from sorted(items)

def uniq_c(items):                   # uniq -c  (adjacent duplicates only!)
    previous, count = None, 0
    for item in items:
        if item == previous:
            count += 1
        else:
            if previous is not None:
                yield count, previous
            previous, count = item, 1
    if previous is not None:
        yield count, previous

def sort_rn(pairs):                  # sort -rn
    yield from sorted(pairs, reverse=True)

def head(n, items):                  # head -n
    for i, item in enumerate(items):
        if i >= n:
            return
        yield item

pipeline = head(3, sort_rn(uniq_c(sort_stage(
    cut(grep("ERROR", cat(LOG_TEXT)), field=3)))))

print("which service produced the most errors?")
for count, service in pipeline:
    print(f"  {count:>3}  {service}")

# The pipeline is lazy: nothing runs until something asks for a value.
lazy = cut(grep("ERROR", cat(LOG_TEXT)), field=3)
print("\nbuilt but not run:", lazy)
print("first value only  :", next(lazy))
print("next value        :", next(lazy))

# And it stays constant-memory: `head` stops the upstream stages early.
def counted(lines, box):
    for line in lines:
        box[0] += 1
        yield line

box = [0]
first_two = list(head(2, counted(cat(LOG_TEXT), box)))
print(f"\nasked for 2 lines; the source produced {box[0]} lines, not "
      f"{len(LOG_TEXT.splitlines())}")

Three properties of that pipeline are the reason to write code this way.

**It is lazy.** Building the chain runs nothing at all — `lazy` prints as a generator object. Each `next()` pulls exactly one value through every stage.

**It short-circuits.** `head(2, ...)` made the source produce three of the eight lines and then stop — three rather than two because `head` has to be *asked* for a third value before it can decide to return, and that request travels upstream. One line of over-read, not five. In a shell this is what makes `head` on a hundred-gigabyte file instant; in Python it is the same mechanism, and it is why `head` belongs at the *end* of the chain rather than being written as a slice of a materialised list.

**`uniq -c` only collapses *adjacent* duplicates**, which is why `sort` has to come first. This is the single most common shell-pipeline bug, and writing the stages as functions makes the reason obvious: `uniq_c` holds one item of state, not a dictionary, so it cannot possibly know about a duplicate it saw twenty lines ago. That is also its virtue — it works on an infinite stream.

When *not* to stream: if you need random access, multiple passes, or the whole dataset in memory anyway, a list is simpler and faster. Generators buy constant memory and early exit; they cost you the ability to look backwards.

## What you built

| Tool | Cost | Use it for | Fails when |
|---|---|---|---|
| adjacency list | $\Theta(V+E)$ space | almost every real graph | the graph is genuinely dense |
| BFS | $O(V+E)$ | levels, shortest path *unweighted* | edges have weights |
| DFS | $O(V+E)$ | reachability, cycles, topological order | you wanted shortest paths |
| Kahn | $O(V+E)$ | dependency order, cycle detection | the graph has a cycle (by design) |
| Dijkstra | $O(E \log V)$ | shortest paths, non-negative weights | any edge is negative |
| Bellman-Ford | $O(V \cdot E)$ | negative edges, arbitrage detection | you needed it to be fast |
| Kruskal + union-find | $O(E \log E)$ | minimum spanning tree | — |
| counting sort | $O(n + k)$ | small integer key range | $k$ is large |
| radix sort | $O(d(n+b))$ | fixed-width integer or string keys | keys are variable-width objects |
| named-group regex | linear, if you avoid nested quantifiers | parsing semi-structured text | the input is really nested (use a parser) |
| generator pipeline | constant memory | streaming transformation | you need random access |

## Try it yourself

Bigger exercises. Every scaffold runs as-is.

### Exercise 1 — Bidirectional BFS

Searching from the start *and* from the target simultaneously, alternating one level each, meets in the middle. If the branching factor is $b$ and the distance is $d$, one-directional BFS explores about $b^d$ vertices and bidirectional about $2b^{d/2}$ — a square-root improvement, which is why route planners use it.

Build a grid graph, run plain BFS and a bidirectional version between two far-apart corners, verify they report the same distance, and print how many vertices each one visited.

In [ ]:
def grid_graph(w, h):
    gg = Graph()
    for x in range(w):
        for y in range(h):
            if x + 1 < w:
                gg.add_edge((x, y), (x + 1, y))
            if y + 1 < h:
                gg.add_edge((x, y), (x, y + 1))
    return gg

def bfs_count(graph, start, target):
    """Plain BFS. Returns (distance, vertices visited)."""
    level, frontier = {start: 0}, deque([start])
    while frontier:
        v = frontier.popleft()
        if v == target:
            return level[v], len(level)
        for nb in graph.neighbours(v):
            if nb not in level:
                level[nb] = level[v] + 1
                frontier.append(nb)
    return None, len(level)

def bidirectional(graph, start, target):
    """Expand one level from each side in turn until the frontiers touch."""
    seen_a, seen_b = {start: 0}, {target: 0}
    fa, fb = deque([start]), deque([target])
    # your code here: alternate expanding one whole level from each side;
    # when a vertex appears in both dicts, the answer is the sum of its depths
    return None, len(seen_a) + len(seen_b)

# Uncomment to test:
# gg = grid_graph(25, 25)
# print("plain BFS      :", bfs_count(gg, (0, 0), (24, 24)))
# print("bidirectional  :", bidirectional(gg, (0, 0), (24, 24)))

### Exercise 2 — A Make-style build planner

`make` is a topological sort plus timestamps. Given targets, their prerequisites, and a modification time for each file, decide which targets are **stale** (a prerequisite is newer than the target, or the target does not exist) and print a build order that rebuilds only what is necessary — remembering that rebuilding a target makes everything depending on it stale too.

Use `kahn` for the order, then walk that order forward propagating staleness. Print the build plan and the skipped targets separately.

In [ ]:
RULES = {                                    # target: [prerequisites]
    "app":      ["main.o", "util.o"],
    "main.o":   ["main.c", "util.h"],
    "util.o":   ["util.c", "util.h"],
    "docs.pdf": ["docs.tex"],
}
MTIME = {"main.c": 10, "util.c": 4, "util.h": 12, "docs.tex": 1,
         "main.o": 11, "util.o": 9, "app": 13, "docs.pdf": 5}

def build_plan(rules, mtime):
    dag = Graph(directed=True)
    for target, prereqs in rules.items():
        for p in prereqs:
            dag.add_edge(p, target)          # prerequisite -> target
    order, ok = kahn(dag)
    assert ok, "circular dependency"
    stale = set()
    # your code here: walk `order`; a target is stale if any prerequisite is
    # stale, or if its mtime is older than a prerequisite's mtime
    return [t for t in order if t in rules and t in stale], stale

# Uncomment to test:
# plan, stale = build_plan(RULES, MTIME)
# print("rebuild in this order:", plan)
# print("up to date           :", [t for t in RULES if t not in stale])

### Exercise 3 — A CSV splitter that respects quotes

Splitting `a,"b,c",d` on commas gives four fields instead of three. The standard-library answer is the `csv` module and you should use it — but writing the pattern once teaches **lookahead**, which is the tool for "a comma, but only if what follows has an even number of quotes":

```text
,(?=(?:[^"]*"[^"]*")*[^"]*$)
```

Read it as: a comma, followed by (zero or more complete quoted pairs, then no more quotes before the end). Test it on the rows below, then compare against `csv.reader` and print any row where the two disagree.

In [ ]:
import csv, io

ROWS = ['a,b,c',
        'a,"b,c",d',
        '"x,y","z",w',
        'plain,"has ""escaped"" quotes",end',
        'trailing,,empty']

FIELD_COMMA = r',(?=(?:[^"]*"[^"]*")*[^"]*$)'

def regex_split(row):
    # your code here: re.split on FIELD_COMMA, then strip surrounding quotes
    return []

# Uncomment to test:
# for row in ROWS:
#     mine = regex_split(row)
#     theirs = next(csv.reader(io.StringIO(row)))
#     flag = "" if mine == theirs else "   <-- DISAGREE"
#     print(f"{row!r:<45} {mine}{flag}")

### Exercise 4 — Stream the log, do not load it

Rewrite section 8's log analysis as a generator pipeline: one stage yields lines from the file, one parses each line into a record (skipping and *counting* failures rather than discarding them silently), one filters to failures, and a final stage tallies by IP. Then assert that the whole chain never holds more than one record at a time — the property that lets the same twenty lines of code process a twenty-gigabyte log.

In [ ]:
def read_lines(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            yield line.rstrip("\n")

def parse_records(lines, bad):
    for line in lines:
        m = LOG.match(line)
        if m is None:
            bad.append(line)
            continue
        # your code here: yield the record dict with status and size as ints
        pass

def only_failures(records):
    # your code here: yield records whose status is >= 400
    return
    yield

# Uncomment to test:
# bad = []
# tally = {}
# for r in only_failures(parse_records(read_lines("access.log"), bad)):
#     tally[r["ip"]] = tally.get(r["ip"], 0) + 1
# print("failures by IP:", tally, " unparseable lines:", len(bad))